In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(host="qdrant", port=6333)
print(client.get_collections())

collections=[CollectionDescription(name='my_docs')]


In [2]:
# Create a collection named "my_docs" 
# (Example using 4-dimensional vectors; your Llama/embedding model will use more, e.g., 4096)
if not client.collection_exists("my_docs"):
    client.create_collection(
        collection_name="my_docs",
        vectors_config=VectorParams(size=4, distance=Distance.COSINE),
    )

In [ ]:
# Prepare your points (records) to write
operation_info = client.upsert(
    collection_name="my_docs",
    wait=True,  # Forces the code to wait until the write is safely completed
    points=[
        PointStruct(
            id=1, 
            vector=[0.05, 0.61, 0.76, 0.24], 
            payload={"title": "Docker Setup", "text": "Running Jupyter and Qdrant."}
        ),
        PointStruct(
            id=2, 
            vector=[0.19, 0.81, 0.33, 0.11], 
            payload={"title": "PyTorch Guide", "text": "How to install PyTorch on CUDA."}
        ),
    ],
)

print("Write Status:", operation_info.status)


read data: 

In [3]:
# Read point ID #1 back from the database
records = client.retrieve(
    collection_name="my_docs",
    ids=[1],
)

for record in records:
    print(f"Read Record {record.id}: {record.payload}")


Read Record 1: {'title': 'Docker Setup', 'text': 'Running Jupyter and Qdrant.'}


read data through search:

In [5]:
# Define a query vector (e.g., generated from a user's question)
query_vector = [0.20, 0.80, 0.30, 0.10]

# Search for the top 1 closest match
search_results = client.query_points(
    collection_name="my_docs",
    query=query_vector,
    limit=1
)

for hit in search_results.points:
    print(f"Match found! ID: {hit.id}, Score: {hit.score}")
    print(f"Document Text: {hit.payload['text']}")


Match found! ID: 2, Score: 0.99946386
Document Text: How to install PyTorch on CUDA.
